In [4]:
import numpy as np
from LanzaModels import TVL1_1D
from ADMMsRustici import MyBackTrackingSolver
from signalClass import *
import time

In [5]:
np.random.seed(24102001)
n = 1024

construct blur matrix

In [6]:
#blur matrix construction

a = 0.25
b = 0.5
c = 0.25

diagB = b * np.ones(shape=(n,))
offDiagA = a * np.ones(shape=(n-1,))
offDiagC = c * np.ones(shape=(n-1,))
A = np.diag(diagB, 0) + np.diag(offDiagC, 1) + np.diag(offDiagA, -1)

#apply anti-reflexive BCs

A[0][0] = 2 * a + b
A[0][1] = c - a
A[n-1][n-2] = a - c
A[n-1][n-1] = b + 2 * c

#end blur matrix construction

construct signal

In [7]:
#begin signal construction

PwSignal = signal(n)
RndSignal = signal(n)
sigma = 0.01

PwSignal.generate_cartoon_sign(2, 100)
RndSignal.generate_GG_realization(0, sigma, 1)

xTrue = PwSignal.get_image()
xCorrupted = (A @ xTrue) + RndSignal.get_image()

#end signal construction

Define the TVL1 model

In [8]:
mu = 1
VarModel = TVL1_1D.TVL1_1DClass(A, xCorrupted, mu)

Now, we need to initialize and define the solver

In [9]:
#begin solver construction

xk = np.copy(xCorrupted)
yk = np.random.randn(n,)
betak = 1
lk = np.zeros(n)

MySolver = MyBackTrackingSolver.MyBacktrackingSolverClass(VarModel, xk, yk, lk, betak)

#end solver construction

In [10]:
iters = 50

XsolutionHistory = np.zeros(shape=(iters, n))
YsolutionHistory = np.zeros(shape=(iters, n))

lambdaHistory = np.zeros(shape=(iters, n))

betaHistory = np.zeros(shape=(iters,))

PrimalResidueHistory = np.zeros(shape=(iters,))
DualResidueHistory = np.zeros(shape=(iters,))

CpuTimes = np.zeros(shape=(iters,))

In [11]:
for iter in range(0, iters):

    print(f"{iter} / {iters}")

    sTime = time.process_time_ns()

    xk_1, yk_1, lk_1, betak_1 = MySolver.CallIterationStep(xk, yk, lk, betak)

    eTime = time.process_time_ns()

    
    primalResidue = np.linalg.norm(VarModel.P @ xk_1 + VarModel.Q @ yk_1 - VarModel.c)
    dualResidue = np.abs(
                (VarModel.mu / 2) * (VarModel.fidelity(xk_1) - VarModel.fidelity(xk) ) + \
                VarModel.regularizer(yk_1) - VarModel.regularizer(yk)							
                        )

    XsolutionHistory[iter, :] = xk
    YsolutionHistory[iter, :] = yk
    lambdaHistory[iter, :] = lk
    betaHistory[iter] = betak

    PrimalResidueHistory[iter] = primalResidue
    DualResidueHistory[iter] = dualResidue
    CpuTimes[iter] = ( (eTime - sTime) / 1e9 ) + CpuTimes[iter - 1]

    xk = xk_1
    yk = yk_1
    lk = lk_1
    betak = betak_1


0 / 50
1 / 50
2 / 50
3 / 50
4 / 50
5 / 50
6 / 50
7 / 50
8 / 50
9 / 50
10 / 50
11 / 50
12 / 50
13 / 50
14 / 50
15 / 50
16 / 50
17 / 50
18 / 50
19 / 50
20 / 50
21 / 50
22 / 50
23 / 50
24 / 50
25 / 50
26 / 50
27 / 50
28 / 50
29 / 50
30 / 50
31 / 50
32 / 50
33 / 50
34 / 50
35 / 50
36 / 50
37 / 50
38 / 50
39 / 50
40 / 50
41 / 50
42 / 50
43 / 50
44 / 50
45 / 50
46 / 50
47 / 50
48 / 50
49 / 50


In [12]:

xReconstr = XsolutionHistory[iters - 1, :]
ConvergenceDistance = np.zeros(shape=(iters,))

ConvergenceDistance = np.linalg.norm(XsolutionHistory - xReconstr, axis=1)

In [13]:
np.savez_compressed(
    "./MyADMMTVL1-Laplace.npz",
    Xs = XsolutionHistory,
    Ys = YsolutionHistory,
    Ls = lambdaHistory,
    Betas = betaHistory,

    PrimalRes = PrimalResidueHistory,
    DualRes = DualResidueHistory,

    ConvergenceDistance = ConvergenceDistance,
    CpuTimes = CpuTimes,

    xTrue = xTrue,
    xCorrupted = xCorrupted,
)